In deze notebook wordt de feature uitgewerkt om de content in huisstijl te uploaden (DFIT-1261).

Om de content opmaak aan te passen maken we eerst een nieuw topic aan, in dit geval een stap. Na het runnen van onderstaande code kun je in cms zien dat content inderdaad in huisstijl is opgemaakt.

In [ ]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.workflow import Workflow
from ask_delphi_api.import_digicoach import Import
from ask_delphi_api.authentication import AskDelphiClient

client = AskDelphiClient()
project = Project(client)
topics = TopicTools(client, project)
workflow = Workflow(client)
import_service = Import()

In [ ]:
# # voor een stap, vul je "Action" als tweede variabel in

# topicId = topics.topic_upload("Nieuwe stap testje2", "Action")
# text = f'<h1>Titel</h1><p>Inleiding met <strong>vet</strong> tekst.</p><ul><li>Eerste punt</li><li>Tweede punt</li></ul>'
# result = topics.checkout(topicId)
# topicVersionId = result['topicVersionId']

# import_service.add_content_to_topic(topicId, topicVersionId, text)

# # Checkin topic.
# topics.checkin(topicId)

# # Topic publiceren
# request_id = workflow.create_workflow_transition_request("c0d52ba6-189e-4efe-a909-74abcca4c0a8")
# transitions_model = workflow.get_workflow_transition_request_transitions_model(request_id)
# workflow.update_workflow_transition_request(request_id, transitions_model)
# workflow.approve_workflow_transition_request(request_id)

Hieronder gaan we onderzoeken of het ook mogelijk is om een taaksjabloon uit een word document te kunnen uploaden waarbij de opmaak van de content wordt meegenomen.

In [ ]:
from ask_delphi_api.convert_taaksjabloon import read_dir, convert_doc_to_json
from pathlib import Path
import json

project_root = Path.cwd().parent
paths = read_dir(project_root/'Taaksjablonen')

json_coaches = []

for path in paths:
    try:
        print("\n\n")
        json_string = convert_doc_to_json(path)

        coach_dict = json.loads(json_string)
        json_coaches.append(coach_dict)
    except ValueError as e:
        print(e)

In het oorspronkelijke document hebben stappen een parent topic, hier wil ik alleen testen of de huisstijl juist wordt doorgevoerd in taken en stappen. Daarom maak ik hier een dummy topic aan voor de voorgedefinieerde zoekopdracht.

In [ ]:
import uuid
checkin_list = []
PREDEFINED_SEARCH_NAME = "VOORGEDEFINIEERDE ZOEKOPDRACHT"

# In dit geval is er maar 1 digicoach, de naam van de digicoach is als volgt:
NAME_DIGICOACH = json_coaches[0]["name"]

# Aanmaken dummy Voorgedefinieerde zoekopdracht topic
topic_id_predefined_search = topics.topic_upload(PREDEFINED_SEARCH_NAME, "Pre-defined search")
topic_version_id_predefined_search = topics.get_topicVersionId(topic_id_predefined_search)
checkin_list.append(("root", topic_id_predefined_search))

# aanmaken procespagina
topic_id_dc = str(uuid.uuid4())
topicTitle = NAME_DIGICOACH
topicTypeId = project.get_topic_type_id("Digitale Coach Procespagina")   

parentTopicRelationTypeId = import_service.relation.get_relation_type_id(topic_id_predefined_search, topic_version_id_predefined_search, "Voorgedefinieerde zoekopdracht")
import_service.relation.add_topic_with_relation(topic_id_dc, topicTitle, topicTypeId, topic_id_predefined_search, parentTopicRelationTypeId, topic_version_id_predefined_search)

# Get Digicoach topic version ID
topic_version_id_digicoach = topics.get_topicVersionId(topic_id_dc)
checkin_list.append(("digicoach", topic_id_dc))

# Tag Digitale Coach Procespagina
import_service.relation.add_tag(topic_id_dc, topic_version_id_digicoach, "interactie")


Hieronder selecteer ik 1 willekeurige taak uit de json en zet deze als topic in het cms

In [ ]:
taken = [task for coach in json_coaches for task in coach["tasks"]]
stappen = [step for coach in json_coaches for task in coach["tasks"] for step in task["steps"]]

reltype_dc_to_task = import_service.relation.get_relation_type_id(
    topic_id_dc,
    topic_version_id_digicoach,
    "Taak"
)

for taak in taken:
    
    task_id = str(uuid.uuid4())
    task_title = taak["name"]
    task_type = project.get_topic_type_id("Task")
    
    import_service.relation.add_topic_with_relation(
        task_id,
        task_title,
        task_type,
        topic_id_dc,
        reltype_dc_to_task,
        topic_version_id_digicoach
    )
    
    task_version_id = topics.get_topicVersionId(task_id)
    checkin_list.append(("task", task_id))
    
    # Description toevoegen
    add_content = taak["description"]
    import_service.add_content_to_topic(task_id, task_version_id, add_content)

    # Stappen
    steps = taak["steps"]
    
    # Relation type Taak → Stap
    reltype_task_to_step = import_service.relation.get_relation_type_id(
        task_id,
        task_version_id,
        "Stap"
    )
    
    for step in steps:
        step_id = str(uuid.uuid4())
        step_title = step["name"]
        step_type = project.get_topic_type_id("Action")
        
        import_service.relation.add_topic_with_relation(
            step_id,
            step_title,
            step_type,
            task_id,
            reltype_task_to_step,
            task_version_id
        )
        
        step_version_id = topics.get_topicVersionId(step_id)
        checkin_list.append(("step", step_id))

        # Beschrijving bij stap
        import_service.add_content_to_topic(
            step_id,
            step_version_id,
            step["description"]
        )

Inchecken van de topics

In [ ]:
priority = {"step": 1, "task": 2, "digicoach": 3, "root": 4}
checkin_list_sorted = sorted(checkin_list, key=lambda x: priority[x[0]])

for typ, tid in checkin_list_sorted:
    topics.checkin(tid)


Publicatie van de topics

In [ ]:
for typ, tid in checkin_list_sorted:
    import_service.publiceer(tid)